In [5]:
import scanpy as sc
import pandas as pd
from pathlib import Path

In [17]:
# Paths & basic setup 
adata_path = "/data/projects/oscar/Miranda/results/adata_brain_microglia_clustering.h5ad"
adata = sc.read_h5ad(adata_path)

In [18]:
# Name of the clustering column in adata.obs
CLUSTER_COL_ADATA = "leiden"   # change if your cluster is named differently

# Make sure clusters are strings for consistent mapping
adata.obs[CLUSTER_COL_ADATA] = adata.obs[CLUSTER_COL_ADATA].astype(str).str.strip()

print("Unique clusters in adata:", sorted(adata.obs[CLUSTER_COL_ADATA].unique()))

Unique clusters in adata: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '4', '5', '6', '7', '8', '9']


In [19]:
# Miranda mapping
cluster_to_type = {
    "0":  "L2/3/4 IT CXT Glut",
    "1":  "L2/3 IT RSP Glut",
    "2":  "Thalamic glutamatergic projection neurons",
    "3":  "L4/5/6 IT CXT Glut",
    "4":  "astrocyte",
    "5":  "L2 IT Glut",              # fixed typo "inteneurons"
    "6":  "microglia",
    "7":  "CA1 ProS",
    "8":  "CA3 Glut",
    "9":  "Excitatory neurons-Slc17a6⁺ glutamatergic projection neurons",
    "10": "activated microglia and macrophage like cell",
    "11": "activated microglia and macrophage like cell",
    "12": "activated microglia and macrophage like cell",
    "13": "activated microglia and macrophage like cell",
    "14": "L6 CT Glut",
    "15": "L5/6 IT CTX Glut",
    "16": "Pvalb Gaba",
    "17": "Vip interneurons",
    "18": "Medium spiny neurons",
    "19": "VLMC",
    "20": "Endothelial cells",
    "21": "PVT Glut",
    "22": "L5 PT Glut",
    "23": "SST interneurons",
    "24": "DG Glut",
    "25": "Oligodendracytes",
    "26": "OPC",
    "27": "Pericytes",
    "28": "VLMC",
    "29": "Choroid Plexus",
    "30": "Antigen-presenting microglia (MHC-II⁺ microglia)",
    "31": "Activated Microglia/Inflammatory microglia"
}

# do we have any clusters in adata not covered by Miranda?
adata_clusters = set(adata.obs[CLUSTER_COL_ADATA].unique())
miranda_clusters = set(cluster_to_type.keys())

missing_in_miranda = adata_clusters - miranda_clusters
extra_in_miranda   = miranda_clusters - adata_clusters

print("Clusters in adata not in Miranda mapping:", missing_in_miranda)
print("Clusters in Miranda mapping not in adata:", extra_in_miranda)

Clusters in adata not in Miranda mapping: set()
Clusters in Miranda mapping not in adata: set()


In [20]:
# Attach annotations to adata.obs 
adata.obs["miranda_cell_type"] = (
    adata.obs[CLUSTER_COL_ADATA]
        .map(cluster_to_type)
        .astype("category")
)

print(adata.obs[[CLUSTER_COL_ADATA, "miranda_cell_type"]].head())
print(adata.obs['miranda_cell_type'].value_counts().to_frame(name='n_cells'))
print("Cells without a mapped type:",
      adata.obs["miranda_cell_type"].isna().sum())

                      leiden   miranda_cell_type
unique_cell_index                               
new_slide1_aaaabngl-1      0  L2/3/4 IT CXT Glut
new_slide1_aaaabpmj-1      0  L2/3/4 IT CXT Glut
new_slide1_aaaadghb-1      5          L2 IT Glut
new_slide1_aaaafgep-1     25    Oligodendracytes
new_slide1_aaaafjjd-1      5          L2 IT Glut
                                                    n_cells
miranda_cell_type                                          
astrocyte                                            129298
Oligodendracytes                                     120764
microglia                                             67481
Pvalb Gaba                                            63905
L2 IT Glut                                            63000
Excitatory neurons-Slc17a6⁺ glutamatergic proje...    61955
Endothelial cells                                     58560
VLMC                                                  41883
Thalamic glutamatergic projection neurons             381

In [22]:
# by cluster
table = (
adata.obs
.groupby([CLUSTER_COL_ADATA, "miranda_cell_type"])
.size()
.reset_index(name='n_cells')
)
pd.set_option("display.max_rows", None)
display(table)

/tmp/ipykernel_757463/2175103764.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby([CLUSTER_COL_ADATA, "miranda_cell_type"])


,leiden,miranda_cell_type,n_cells
0,0,Activated Microglia/Inflammatory microglia,0
1,0,Antigen-presenting microglia (MHC-II⁺ microglia),0
2,0,CA1 ProS,0
3,0,CA3 Glut,0
4,0,Choroid Plexus,0
5,0,DG Glut,0
6,0,Endothelial cells,0
7,0,Excitatory neurons-Slc17a6⁺ glutamatergic proj...,0
8,0,L2 IT Glut,0
9,0,L2/3 IT RSP Glut,0


In [23]:
# Save updated AnnData 
out_h5ad = adata_path.replace(".h5ad", "_with_mirandaUPDATE.h5ad")
adata.write(out_h5ad)
print("Saved updated AnnData with Miranda annotations:", out_h5ad)

Saved updated AnnData with Miranda annotations: /data/projects/oscar/Miranda/results/adata_brain_microglia_clustering_with_mirandaUPDATE.h5ad


In [24]:
import os
import pandas as pd
import scanpy as sc

adata_path = "/data/projects/oscar/Miranda/results/adata_brain_microglia_clustering_with_mirandaUPDATE.h5ad"
out_dir = "/data/projects/oscar/Miranda/results"
os.makedirs(out_dir, exist_ok=True)

adata = sc.read_h5ad(adata_path)
obs = adata.obs.copy()

# Decide which column identifies slide/run
run_col = None
for c in ["run_key", "batch", "sample_id"]:
    if c in obs.columns:
        run_col = c
        break

if run_col is None:
    # fallback: infer run from obs_names like "new_slide1_12345"
    obs["__run__"] = pd.Index(obs.index).to_series().astype(str).str.split("_", n=1).str[0].values
    run_col = "__run__"

print("Using run column:", run_col)
print("Runs found:", obs[run_col].unique().tolist())

#    Make sure we have a cell_id column for Xenium Explorer
#    Important: Xenium usually wants the *original* cell_id per run,
#    not the namespaced one. We'll strip the "run_" prefix if present.
if "cell_id" in obs.columns:
    cell_id_series = obs["cell_id"].astype(str)
else:
    cell_id_series = pd.Index(obs.index).astype(str)

# If obs_names are like "new_slide1_12345", strip prefix to get "12345"
# (only strip if it looks like it starts with "<run>_")
def strip_prefix(cell_id, run):
    s = str(cell_id)
    pref = f"{run}_"
    return s[len(pref):] if s.startswith(pref) else s

# Export per run
required_col = "miranda_cell_type" ########################################################
if required_col not in obs.columns:
    raise ValueError(f"Missing '{required_col}' in adata.obs. Columns: {obs.columns.tolist()}")

for run in sorted(obs[run_col].astype(str).unique()):
    sub = obs[obs[run_col].astype(str) == run].copy()

    # build per-run cell_id
    sub_cell_id = (sub["cell_id"].astype(str) if "cell_id" in sub.columns else pd.Index(sub.index).astype(str))
    sub_cell_id = [strip_prefix(cid, run) for cid in sub_cell_id]

    export = pd.DataFrame({
        "cell_id": sub_cell_id,
        "group": sub[required_col].astype(str).values
    })

    out_path = os.path.join(out_dir, f"cell_groups_mirandaUPDATE_{run}.csv")
    export.to_csv(out_path, index=False)
    print("Wrote:", out_path, "| rows:", export.shape[0])

Using run column: run_key
Runs found: ['new_slide1', 'new_slide2']
Wrote: /data/projects/oscar/Miranda/results/cell_groups_mirandaUPDATE_new_slide1.csv | rows: 504697
Wrote: /data/projects/oscar/Miranda/results/cell_groups_mirandaUPDATE_new_slide2.csv | rows: 443917
